# Rock quarterly processing — physical then NewMedia

Runs in order:

1. **Physical** — append TYPE rows from the TWN/HK/MAL/CHINA PDFs (excluding NEW) to the consolidated physical workbook and save it in `_output`.
2. **NewMedia** — clean the NewMedia Excel files, update `Rock_lookup_fx.csv`, append this quarter’s physical rows from **that `_output` file only**, write `MOK 2026Q1_JN.xlsx`, and compare `Royalty (HKD)` to the 總表 PDF.

Use **Run All**. One timestamped log is written to the output folder.

In [17]:
import os
import re
import sys
import csv
import shutil
import zipfile
from datetime import datetime
from collections import Counter, OrderedDict
from copy import copy as copy_style

from openpyxl import load_workbook, Workbook
from openpyxl.formula.translate import Translator
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
REPORT_YEAR = 2026
REPORT_QUARTER = 'Q2'
REV_YEAR = 2026
REV_QUARTER = 'Q2'

karen_root = '/Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen'
output_root = '/Users/johannesnatterer/Developer/_output'

consolidated_dir = f'{karen_root}/Rock Music/Consolidated statements/'
quarterly_dir = f'{karen_root}/Rock Music/Quarterly Statements/2026 Q2/as sent by Rock'
lookup_dir = f'{karen_root}/Rock Music/Consolidated statements/lookup_tables/'
outputdirectory = f'{output_root}/'

physical_input_file = 'old/ZZ_Rock_royalties_physical_2024Q1_2026Q1.xlsx'
physical_output_file = 'ZZ_Rock_royalties_physical_2024Q1_2026Q2.xlsx'
file_newmedia = 'MOK 2026Q2_NewMeidia.xlsx'
lookup_fx_file = 'Rock_lookup_fx.csv'
summary_pdf = '2026Q2 MOK-總表.pdf'
newmedia_output_file = 'MOK 2026Q2_JN.xlsx'

filename_snippets = ['TWN', 'HK', 'MAL', 'CHINA']
filemustnotcontain = 'NEW'
CURRENCY_BY_SNIPPET = {'HK': 'HKD', 'MAL': 'MAL', 'TWN': 'NTD', 'CHINA': 'RMB'}
SNIPPETS_LONGEST_FIRST = sorted(filename_snippets, key=len, reverse=True)

DATA_SHEET = 'data'
MAFORMA_SHEET = 'data - converted to maforma'

COL_CATALOG = 2
COL_TITLE = 4
COL_REL_DATE = 5
COL_TYPE = 6
COL_UNITS = 9
COL_WS = 10
COL_BASE = 11
COL_PRORATA = 12
COL_CTRL = 13
COL_RATE = 14
COL_GROSS = 15
COL_CURRENCY = 16
COL_REP_YEAR = 17
COL_REP_QTR = 18
COL_REV_YEAR = 19
COL_REV_QTR = 20
COL_NUM = 22
COL_DEN = 23
COL_TERRITORY = 21
GROSS_FORMULA = '=+K{r}*V{r}/W{r}*I{r}*N{r}/100'

CURRENCY_PAIRS = [
    'Currency:HKD/HKD', 'Currency:NTD/HKD', 'Currency:USD/HKD',
    'Currency:RMB/HKD', 'Currency:SGD/HKD', 'Currency:YEN/HKD', 'Currency:MYR/HKD',
]
FX_WRITE_ORDER = ['NTD', 'RMB', 'SGD', 'USD', 'YEN', 'HKD', 'MYR']
CURRENCY_RE = re.compile(r'^Currency:([A-Za-z]+)/HKD\s*$', re.I)
RATIO_RE = re.compile(r'^\s*([0-9]+(?:\.[0-9]+)?)\s*[:/]\s*([0-9]+(?:\.[0-9]+)?)\s*$')
PDF_CURRENCY_RE = re.compile(
    r'Currency:([A-Za-z]+)/HKD\s*([0-9]+(?:\.[0-9]+)?)\s*[:/]\s*([0-9]+(?:\.[0-9]+)?)', re.I)
PDF_SUBTOTAL_RE = re.compile(
    r'\[HKD:([A-Za-z]+)\]\s*([0-9]+(?:\.[0-9]+)?)\s*:\s*([0-9]+(?:\.[0-9]+)?)', re.I)

NEW_HEADERS = [
    'Entry No.', 'Payer/Licensee', 'Payee/Licensor', 'Territory',
    'Report year', 'Report Quarter', 'Rev Year', 'Rev Quarter',
]
EXTRA_HEADERS = ['WS Price', 'Base Price', 'Release Date', 'Type']


def resolve_dir(rel):
    cwd = os.getcwd()
    alt = rel.replace('../../', '../', 1) if rel.startswith('../../') else rel
    candidates = [
        os.path.abspath(os.path.join(cwd, rel)),
        os.path.abspath(os.path.join(cwd, 'Karen_statements', rel)),
        os.path.abspath(os.path.join(cwd, alt)),
        os.path.abspath(os.path.join(os.path.dirname(cwd), rel)),
        os.path.abspath(os.path.join(cwd, '..', alt)),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f'Directory not found: {rel}\nTried:\n- ' + '\n- '.join(candidates))


consolidated_dir = resolve_dir(consolidated_dir)
quarterly_dir = resolve_dir(quarterly_dir)
lookup_dir = resolve_dir(lookup_dir)

physical_input_path = os.path.join(consolidated_dir, physical_input_file)
physical_output_path = os.path.join(outputdirectory, physical_output_file)
path_newmedia = os.path.join(quarterly_dir, file_newmedia)
path_fx = os.path.join(lookup_dir, lookup_fx_file)
path_summary_pdf = os.path.join(quarterly_dir, summary_pdf)
newmedia_output_path = os.path.join(outputdirectory, newmedia_output_file)

os.makedirs(outputdirectory, exist_ok=True)
logfile_name = (
    os.path.splitext(newmedia_output_file)[0]
    + '_run_log_'
    + datetime.now().strftime('%Y%m%d_%H%M%S') + '.txt'
)
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout


class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()
    def flush(self):
        for s in self.streams:
            s.flush()
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.streams[0], name)


sys.stdout = _Tee(_original_stdout, _log_file)


def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f'Run log saved to: {log_path}')


def fmt_int(n):
    try:
        return f'{int(n):,}'
    except (TypeError, ValueError):
        return str(n)


def fmt_money(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def header(title):
    print(f"\n{'=' * 72}\n  {title}\n{'=' * 72}")


def subheader(title):
    print(f'\n--- {title} ---')


def as_year(value):
    s = str(value).strip()
    return int(s) if s.isdigit() else s


def as_number(value):
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value


def to_float(value):
    if value is None or value == '':
        return None
    if isinstance(value, str) and value.startswith('='):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


header('Rock quarterly processing')
print(f'  Run started              : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Report period            : {REPORT_YEAR} {REPORT_QUARTER}')
print(f'  Physical input           : {physical_input_file}')
print(f'  Physical output          : {physical_output_file}')
print(f'  NewMedia output          : {newmedia_output_file}')
print(f'  Quarterly dir            : {quarterly_dir}')
print(f'  Output dir               : {outputdirectory}')
print(f'  Run log                  : {logfile_name}')

header('0. Required input files')
required = [physical_input_path, path_newmedia, path_fx, path_summary_pdf]
missing = []
for p in required:
    if os.path.isfile(p):
        print(f'  [OK]       {p}')
    else:
        print(f'  [MISSING]  {p}')
        missing.append(p)
if missing:
    close_log()
    raise FileNotFoundError('Stopping: required file(s) not found:\n- ' + '\n- '.join(missing))
print(f'  [OK]       PDF/NewMedia folder: {quarterly_dir}')


  Rock quarterly processing
  Run started              : 2026-09-24 11:08:52
  Report period            : 2026 Q2
  Physical input           : old/ZZ_Rock_royalties_physical_2024Q1_2026Q1.xlsx
  Physical output          : ZZ_Rock_royalties_physical_2024Q1_2026Q2.xlsx
  NewMedia output          : MOK 2026Q2_JN.xlsx
  Quarterly dir            : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/Rock Music/Quarterly Statements/2026 Q2/as sent by Rock
  Output dir               : /Users/johannesnatterer/Developer/_output/
  Run log                  : MOK 2026Q2_JN_run_log_20260924_110852.txt

  0. Required input files
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/Rock Music/Consolidated statements/old/ZZ_Rock_royalties_physical_2024Q1_2026Q1.xlsx
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen

In [18]:
CATALOG_RE = re.compile(r'^[A-Za-z][A-Za-z0-9.\-]*\d[A-Za-z0-9.\-]*$')
NUMERIC_CATALOG_RE = re.compile(r'^(?P<catalog>\d[\d.\-]*)(?P<rest>.*)$')
ROW_RE = re.compile(
    r'^(?P<head>.*?)'
    r'(?P<date>\d{4}/\d{2}/\d{2})\s+'
    r'(?P<units>-?[\d,]+)\s+'
    r'\$(?P<ws>[\d,]+\.\d{2})\s+'
    r'\$(?P<gross>-?[\d,]+\.\d{2})\s+'
    r'\$(?P<rest>\S+)\s+'
    r'(?P<ctrl>[\d.]+)\s+'
    r'(?P<rate_type>\S+)\s*$'
)
STATED_CCY_RE = re.compile(r'(?<![:/\w])Currency:\s*([A-Za-z]+)(?!\s*/)')
TERRITORY_BY_SNIPPET = {
    'HK': 'Hong Kong', 'MAL': 'Malaysia', 'TWN': 'Taiwan', 'CHINA': 'China',
}
TERRITORY_BY_CURRENCY = {
    'HKD': 'Hong Kong', 'NTD': 'Taiwan', 'RMB': 'China', 'CNY': 'China',
    'MYR': 'Malaysia', 'MAL': 'Malaysia',
}


def split_catalog_and_title(head, last_title):
    """Letter codes (RD20854, RCD-0004) or digit codes (1760516, 5051442393125)."""
    head = (head or '').strip()
    if not head:
        return None, last_title
    parts = head.split(None, 1)
    if CATALOG_RE.match(parts[0]):
        title = parts[1].strip() if len(parts) > 1 and parts[1].strip() else last_title
        return parts[0], title
    m = NUMERIC_CATALOG_RE.match(head)
    if m and sum(ch.isdigit() for ch in m.group('catalog')) >= 5:
        rest = m.group('rest')
        if rest == '' or rest[0].isspace() or not rest[0].isascii():
            title = rest.strip() or last_title
            return m.group('catalog'), title
    return None, head


def parse_money(s):
    return float(str(s).replace(',', ''))


def snippet_from_filename(name):
    upper = name.upper()
    for snippet in SNIPPETS_LONGEST_FIRST:
        if snippet.upper() in upper:
            return snippet
    return None


def currency_from_filename(name):
    snippet = snippet_from_filename(name)
    if snippet is None:
        return None, None
    return snippet, CURRENCY_BY_SNIPPET[snippet]


def split_base_prorata(token, units, rate, gross, ws):
    """PDF concatenates Base Price and Song Pro Rata (e.g. $105.0001/22)."""
    if '/' not in token:
        return None
    left, den_s = token.rsplit('/', 1)
    try:
        den = int(den_s)
    except ValueError:
        return None
    if den == 0 or '.' not in left:
        return None
    intpart, frac = left.split('.', 1)
    best = None
    for k in range(1, len(frac) + 1):
        num = int(frac[-k:])
        if num == 0:
            continue
        base_frac = frac[:-k]
        base = float(intpart + ('.' + base_frac if base_frac else ''))
        calc = base * num / den * units * rate / 100.0
        rec = (abs(calc - gross), abs(base - ws), -k, base, num, den)
        if best is None or rec < best:
            best = rec
    if best is None:
        return None
    err, _, _, base, num, den = best
    return base, num, den, err


def parse_pdf_physical_rows(path):
    reader = PdfReader(str(path))
    text = '\n'.join((page.extract_text() or '') for page in reader.pages)
    stated = {m.group(1).upper() for m in STATED_CCY_RE.finditer(text)}
    rows = []
    skipped_no_type = 0
    skipped_zero = 0
    last_catalog = None
    last_title = None
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        m = ROW_RE.search(line)
        if not m:
            continue
        rate_type = m.group('rate_type')
        rm = re.match(r'^(\d+\.\d{2})(.*)$', rate_type)
        if not rm:
            skipped_no_type += 1
            continue
        rate = float(rm.group(1))
        typ = rm.group(2).strip()
        if not typ:
            skipped_no_type += 1
            continue
        units = int(m.group('units').replace(',', ''))
        ws = parse_money(m.group('ws'))
        gross = parse_money(m.group('gross'))
        ctrl = float(m.group('ctrl'))
        split = split_base_prorata(m.group('rest'), units, rate, gross, ws)
        if not split:
            raise ValueError(f'Could not split Base Price / Song Pro Rata in: {line[:200]}')
        base, num, den, err = split
        head = m.group('head').strip()
        catalog, title = split_catalog_and_title(head, last_title)
        if catalog:
            last_catalog = catalog
        else:
            catalog = last_catalog
        if title:
            last_title = title
        else:
            title = last_title
        if gross == 0:
            skipped_zero += 1
            continue
        rel = datetime.strptime(m.group('date'), '%Y/%m/%d')
        rows.append({
            'catalog': catalog,
            'title': title,
            'rel': rel,
            'type': typ,
            'units': units,
            'ws': as_number(ws),
            'base': as_number(base),
            'prorata': f'{num}/{den}',
            'num': num,
            'den': den,
            'ctrl': as_number(ctrl),
            'rate': as_number(rate),
            'gross_pdf': gross,
            'gross_err': err,
        })
    return rows, skipped_no_type, skipped_zero, len(reader.pages), stated


def last_data_row(ws, max_col=None):
    last = 1
    ncols = max_col or ws.max_column or 1
    for r in range(ws.max_row, 1, -1):
        if any(ws.cell(r, c).value not in (None, '') for c in range(1, ncols + 1)):
            last = r
            break
    return last


def pull_down_row(ws, src_row, dest_row):
    """Copy one sheet row, translating formulas to the destination row."""
    max_col = ws.max_column or 1
    for c in range(1, max_col + 1):
        src = ws.cell(src_row, c)
        dest = ws.cell(dest_row, c)
        val = src.value
        if isinstance(val, str) and val.startswith('='):
            dest.value = Translator(val, origin=src.coordinate).translate_formula(dest.coordinate)
        else:
            dest.value = val
        dest.number_format = src.number_format
        if src.has_style:
            dest.font = copy_style(src.font)
            dest.border = copy_style(src.border)
            dest.fill = copy_style(src.fill)
            dest.alignment = copy_style(src.alignment)
            dest.protection = copy_style(src.protection)


header('PART 1 — Physical royalties')
header('1. Select PDF files')
all_pdfs = sorted(
    f for f in os.listdir(quarterly_dir)
    if f.lower().endswith('.pdf') and not f.startswith('.')
)
print(f'  PDFs in folder           : {fmt_int(len(all_pdfs))}')

selected = []
skipped = []
for name in all_pdfs:
    upper = name.upper()
    snippet = snippet_from_filename(name)
    if snippet is None:
        skipped.append((name, 'no TWN/HK/MAL/CHINA snippet'))
        continue
    if filemustnotcontain.upper() in upper:
        skipped.append((name, f'contains "{filemustnotcontain}"'))
        continue
    snippet_key, ccy = currency_from_filename(name)
    selected.append((name, snippet_key, ccy))

print(f'  Qualifying PDFs          : {fmt_int(len(selected))}')
print(f'  Skipped PDFs             : {fmt_int(len(skipped))}')
for name, snippet_key, ccy in selected:
    print(f'  [USE]      {name:<45}  snippet={snippet_key:<6}  currency={ccy}')
for name, reason in skipped:
    print(f'  [SKIP]     {name:<45}  {reason}')

if not selected:
    close_log()
    raise FileNotFoundError('No qualifying PDF files found.')

header('2. Read qualifying PDFs (TYPE rows only)')
new_rows = []
for name, snippet_key, ccy in selected:
    path = os.path.join(quarterly_dir, name)
    subheader(name)
    rows, skipped_no_type, skipped_zero, pages, stated = parse_pdf_physical_rows(path)
    print(f'  Pages                    : {pages}')
    print(f'  Rows with TYPE           : {fmt_int(len(rows))}')
    print(f'  TYPE rows, gross = 0     : {fmt_int(skipped_zero)}  (not read)')
    print(f'  Rows without TYPE        : {fmt_int(skipped_no_type)}  (digital / skipped)')
    if len(stated) == 1:
        stated_ccy = next(iter(stated))
        if stated_ccy != ccy:
            print(
                f'  Currency on statement    : {stated_ccy}  '
                f'(filename snippet {snippet_key} maps to {ccy})'
            )
            ccy = stated_ccy
        else:
            print(f'  Currency on statement    : {stated_ccy}')
    else:
        print(f'  Currency from filename   : {ccy}')
    pdf_gross = sum(r['gross_pdf'] for r in rows)
    pdf_units = sum(r['units'] for r in rows)
    print(f'  TYPE units               : {fmt_int(pdf_units)}')
    print(f'  TYPE gross (PDF)         : {fmt_money(pdf_gross)} {ccy}')
    missing_catalog = sum(1 for r in rows if not r['catalog'])
    if missing_catalog:
        print(f'  WARNING: {fmt_int(missing_catalog)} TYPE row(s) have no catalog after forward-fill')
    max_err = max((r['gross_err'] for r in rows), default=0)
    if rows:
        print(f'  Max gross parse error    : {max_err:.4f}')
    for r in rows:
        r['source_file'] = name
        r['snippet'] = snippet_key
        r['currency'] = ccy
        r['territory'] = TERRITORY_BY_SNIPPET[snippet_key]
        new_rows.append(r)
        print(
            f'    {str(r["catalog"] or ""):<12} {str(r["title"] or "")[:32]:<32} '
            f'{r["type"]:<5} {fmt_int(r["units"]):>8}  {r["prorata"]:<8}  '
            f'{fmt_money(r["gross_pdf"]):>12} {ccy}'
        )

print()
print(f'  Total TYPE rows to append: {fmt_int(len(new_rows))}')
by_ccy = Counter((r['currency'], r['source_file']) for r in new_rows)
for (ccy, src), n in sorted(by_ccy.items()):
    print(f'    {n:>4}  {ccy:<4}  {src}')

header('3. Append to data tab')
print(f'  Copying workbook to output: {physical_output_path}')
shutil.copy2(physical_input_path, physical_output_path)

wb = load_workbook(physical_output_path)
if DATA_SHEET not in wb.sheetnames:
    close_log()
    raise KeyError(f'Sheet "{DATA_SHEET}" not found. Sheets: {wb.sheetnames}')
ws = wb[DATA_SHEET]

if ws.cell(1, COL_NUM).value in (None, ''):
    ws.cell(1, COL_NUM).value = 'Song pro Rata nominator'
if ws.cell(1, COL_DEN).value in (None, ''):
    ws.cell(1, COL_DEN).value = 'Song pro Rata denominator'

rep_year = as_year(REPORT_YEAR)
rev_year = as_year(REV_YEAR)
last = last_data_row(ws)
print(f'  data sheet last used row : {fmt_int(last)}')
print(f'  Sheets                   : {wb.sheetnames}')

date_format = 'mm-dd-yy'
for sample_r in range(2, last + 1):
    fmt = ws.cell(sample_r, COL_REL_DATE).number_format
    if fmt and fmt != 'General':
        date_format = fmt
        break

start_row = last + 1
for i, rec in enumerate(new_rows):
    r = start_row + i
    ws.cell(r, COL_CATALOG).value = rec['catalog']
    ws.cell(r, COL_TITLE).value = rec['title']
    cell_date = ws.cell(r, COL_REL_DATE)
    cell_date.value = rec['rel']
    cell_date.number_format = date_format
    ws.cell(r, COL_TYPE).value = rec['type']
    ws.cell(r, COL_UNITS).value = rec['units']
    ws.cell(r, COL_WS).value = rec['ws']
    ws.cell(r, COL_BASE).value = rec['base']
    ws.cell(r, COL_PRORATA).value = rec['prorata']
    ws.cell(r, COL_CTRL).value = rec['ctrl']
    ws.cell(r, COL_RATE).value = rec['rate']
    ws.cell(r, COL_GROSS).value = GROSS_FORMULA.format(r=r)
    ws.cell(r, COL_CURRENCY).value = rec['currency']
    ws.cell(r, COL_TERRITORY).value = rec['territory']
    ws.cell(r, COL_REP_YEAR).value = rep_year
    ws.cell(r, COL_REP_QTR).value = REPORT_QUARTER
    ws.cell(r, COL_REV_YEAR).value = rev_year
    ws.cell(r, COL_REV_QTR).value = REV_QUARTER
    ws.cell(r, COL_NUM).value = rec['num']
    ws.cell(r, COL_DEN).value = rec['den']

end_row = start_row + len(new_rows) - 1 if new_rows else last
print(f'  Appended rows            : {fmt_int(len(new_rows))}  (Excel rows {start_row}–{end_row if new_rows else last})')
print(f'  Gross Royalty formula    : {GROSS_FORMULA}')
print(f'  Song Pro Rata            : text fraction in L; nominator V / denominator W')

header('4. Pull down maforma formulas')
if MAFORMA_SHEET not in wb.sheetnames:
    close_log()
    raise KeyError(f'Sheet "{MAFORMA_SHEET}" not found. Sheets: {wb.sheetnames}')
ws_m = wb[MAFORMA_SHEET]
maforma_last = last_data_row(ws_m)
print(f'  maforma last used row    : {fmt_int(maforma_last)}')
if new_rows:
    for i in range(1, len(new_rows) + 1):
        pull_down_row(ws_m, maforma_last, maforma_last + i)
        terr = ws_m.cell(maforma_last + i, 4)
        if isinstance(terr.value, str):
            terr.value = terr.value.replace('"CNY"', '"RMB"')
        rec = new_rows[i - 1]
        inferred = TERRITORY_BY_CURRENCY.get(str(rec['currency']).upper())
        if rec.get('territory') and rec['territory'] != inferred:
            terr.value = rec['territory']
    maforma_end = maforma_last + len(new_rows)
    print(f'  Formulas pulled down     : {fmt_int(len(new_rows))} rows  (Excel rows {maforma_last + 1}–{maforma_end})')
    print(f'  Template row             : {maforma_last}')
    sample = ws_m.cell(maforma_end, 4).value
    print(f'  Sample Territory formula : {sample}')
else:
    print('  No new rows — maforma unchanged.')

if hasattr(wb, 'calculation') and wb.calculation is not None:
    wb.calculation.calcMode = 'auto'
    wb.calculation.fullCalcOnLoad = True

wb.save(physical_output_path)
wb.close()

header('5. Physical output summary')
print(f'  Workbook written         : {physical_output_path}')
print(f'  TYPE rows added          : {fmt_int(len(new_rows))}')
print(f'  maforma formulas extended: {fmt_int(len(new_rows))} rows')
by_currency = Counter(r['currency'] for r in new_rows)
print('  By currency:')
for ccy, n in sorted(by_currency.items()):
    units = sum(r['units'] for r in new_rows if r['currency'] == ccy)
    gross = sum(r['gross_pdf'] for r in new_rows if r['currency'] == ccy)
    print(f'    {ccy:<4}  rows {fmt_int(n):>4}    units {fmt_int(units):>8}    PDF gross {fmt_money(gross):>12}')
print('  Refresh the Pivot sheet in Excel after opening the file.')
print(f'  Part 1 finished          : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')



  PART 1 — Physical royalties

  1. Select PDF files
  PDFs in folder           : 7
  Qualifying PDFs          : 4
  Skipped PDFs             : 3
  [USE]      2026Q2 MOK-CHINA.pdf                           snippet=CHINA   currency=RMB
  [USE]      2026Q2 MOK-HK.pdf                              snippet=HK      currency=HKD
  [USE]      2026Q2 MOK-MAL.pdf                             snippet=MAL     currency=MAL
  [USE]      2026Q2 MOK-TWN.pdf                             snippet=TWN     currency=NTD
  [SKIP]     2026Q2 MOK-HK_NewMedia.pdf                     contains "NEW"
  [SKIP]     2026Q2 MOK-TWN_ NewMedia.pdf                   contains "NEW"
  [SKIP]     2026Q2 MOK-總表.pdf                              no TWN/HK/MAL/CHINA snippet

  2. Read qualifying PDFs (TYPE rows only)

--- 2026Q2 MOK-CHINA.pdf ---
  Pages                    : 1
  Rows with TYPE           : 2
  TYPE rows, gross = 0     : 0  (not read)
  Rows without TYPE        : 0  (digital / skipped)
  Currency on statement    :

In [19]:
def cell_str(value):
    if value is None:
        return ''
    return str(value).strip()


def after_colon(value):
    text = cell_str(value)
    if ':' not in text:
        return None
    return text.split(':', 1)[1].strip()


def territory_from_sheet(sheet_name):
    upper = sheet_name.upper()
    if 'TWN' in upper:
        return 'Taiwan'
    if 'MAL' in upper:
        return 'Malaysia'
    if 'HK' in upper:
        return 'Hong Kong'
    return None


def parse_ratio(value):
    m = RATIO_RE.match(cell_str(value).replace(',', ''))
    if not m:
        return None
    left, right = float(m.group(1)), float(m.group(2))
    if right == 0:
        return None
    return left, right, left / right


def format_rate(x):
    x = round(float(x), 9)
    if abs(x - round(x)) < 1e-12:
        return str(int(round(x)))
    return f'{x:.9f}'.rstrip('0').rstrip('.')


def format_pair_number(x):
    if abs(x - round(x)) < 1e-12:
        return str(int(round(x)))
    return f'{x:.10f}'.rstrip('0').rstrip('.')


def clean_source_header(name):
    text = cell_str(name)
    if text == 'SHARE AMOUNT':
        return 'SHARE AMOUNT (local FX)'
    return text


def add_fx_hit(store, code, left, right, rate, source):
    code = code.upper()
    rec = {
        'currency': code,
        'left': left,
        'right': right,
        'rate': rate,
        'pair': f'{format_pair_number(left)}:{format_pair_number(right)}',
        'source': source,
    }
    if code in store and store[code]['pair'] != rec['pair']:
        print(
            f'  WARNING: {code} already {store[code]["pair"]} from {store[code]["source"]}; '
            f'keeping first, ignoring {rec["pair"]} from {source}'
        )
        return
    if code not in store:
        store[code] = rec


def scan_pdfs_for_fx(folder, store):
    max_pdf_bytes = 1 * 1024 * 1024  # 1 MB
    for name in sorted(os.listdir(folder)):
        if not name.lower().endswith('.pdf'):
            continue
        path = os.path.join(folder, name)
        size = os.path.getsize(path)
        if size > max_pdf_bytes:
            print(f'  [SKIP]     {name}  ({size / (1024 * 1024):.1f} MB > 1 MB)')
            continue
        try:
            reader = PdfReader(path)
            text = '\n'.join((page.extract_text() or '') for page in reader.pages)
        except Exception as exc:
            print(f'  WARNING: could not read PDF {name}: {exc}')
            continue
        for m in PDF_CURRENCY_RE.finditer(text):
            left, right = float(m.group(2)), float(m.group(3))
            if right:
                add_fx_hit(store, m.group(1), left, right, left / right, f'PDF {name}')
        for m in PDF_SUBTOTAL_RE.finditer(text):
            hkd, other = float(m.group(2)), float(m.group(3))
            code = m.group(1).upper()
            if hkd:
                add_fx_hit(store, code, other, hkd, other / hkd, f'PDF {name} [HKD:{code}]')


SOURCE_COLUMNS = [
    'USER', 'CATALOG NO.', 'ISRC', 'PayType', 'CATALOG TITLE', 'SONG TITLE',
    'ARTIST', 'REVENUE PERIOD', 'UNIT', 'AMOUNT', 'SongProRata', 'Ctrl.%',
    'Royalty Rate%', 'ROYALTY', 'Share%', 'SHARE AMOUNT (local FX)', 'Currency',
]


def process_sheet(ws, sheet_name, source_file, fx_store):
    territory = territory_from_sheet(sheet_name)
    if territory is None:
        raise ValueError(f'Cannot derive territory from sheet name {sheet_name!r}')

    payer = None
    payee = None
    header_row_idx = None
    source_headers = None
    data_rows = []
    n_read = 0
    n_empty_a = 0
    n_above_user = 0
    unknown_headers = []

    for i, row in enumerate(ws.iter_rows(min_row=1, values_only=True), start=1):
        n_read += 1
        cells = list(row)
        col_a = cell_str(cells[0] if cells else None)

        for j, value in enumerate(cells):
            m = CURRENCY_RE.match(cell_str(value))
            if not m:
                continue
            rate_cell = cells[j + 1] if j + 1 < len(cells) else None
            parsed = parse_ratio(rate_cell)
            label = f'Currency:{m.group(1).upper()}/HKD'
            if parsed is None:
                print(f'  WARNING: {source_file} {sheet_name} r{i} {label} has unreadable rate {rate_cell!r}')
            else:
                left, right, rate = parsed
                add_fx_hit(
                    fx_store, m.group(1), left, right, rate,
                    f'{source_file} / {sheet_name} r{i}',
                )

        if col_a.startswith('Payer/Licensee:'):
            payer = after_colon(col_a)
        elif col_a.startswith('Payee/Licensor:'):
            payee = after_colon(col_a)

        if header_row_idx is None:
            if col_a.upper() == 'USER':
                header_row_idx = i
                source_headers = [clean_source_header(v) for v in cells]
                while source_headers and source_headers[-1] == '':
                    source_headers.pop()
                unknown_headers = [
                    h for h in source_headers if h and h not in SOURCE_COLUMNS
                ]
                n_above_user = i - 1
            continue

        if col_a == '':
            n_empty_a += 1
            continue

        values = {}
        for idx, header in enumerate(source_headers or []):
            if not header:
                continue
            values[header] = cells[idx] if idx < len(cells) else None
        prefix = [
            None,
            payer,
            payee,
            territory,
            REPORT_YEAR,
            REPORT_QUARTER,
            REV_YEAR,
            REV_QUARTER,
        ]
        aligned = [values.get(name) for name in SOURCE_COLUMNS]
        data_rows.append(prefix + aligned + [None, None, None, None])

    if unknown_headers:
        print(f'  WARNING: {source_file} / {sheet_name} unexpected columns: {unknown_headers}')

    return {
        'source_file': source_file,
        'sheet_name': sheet_name,
        'territory': territory,
        'payer': payer,
        'payee': payee,
        'header_row_idx': header_row_idx,
        'source_headers': source_headers,
        'data_rows': data_rows,
        'n_read': n_read,
        'n_empty_a': n_empty_a,
        'n_above_user': n_above_user,
    }


def parse_fraction(value):
    text = cell_str(value).lstrip('=+')
    if '/' not in text:
        return None
    left, right = text.split('/', 1)
    try:
        num = int(float(left))
        den = int(float(right))
    except ValueError:
        return None
    if den == 0:
        return None
    return num, den


def territory_from_currency(currency):
    code = cell_str(currency).upper()
    if code == 'HKD':
        return 'Hong Kong'
    if code == 'NTD':
        return 'Taiwan'
    if code in ('CNY', 'RMB'):
        return 'China'
    if code in ('MYR', 'MAL'):
        return 'Malaysia'
    return None


def compute_physical_gross(units, base, rate, num, den):
    try:
        units_n = float(units)
        base_n = float(base)
        rate_n = float(rate)
        num_n = float(num)
        den_n = float(den)
    except (TypeError, ValueError):
        return None
    if den_n == 0:
        return None
    return base_n * num_n / den_n * units_n * rate_n / 100.0


def physical_data_row_to_output(values, out_headers, payer, payee):
    """Build a JN row from the physical `data` tab (formulas are not yet cached)."""
    catalog = values[COL_CATALOG - 1] if len(values) >= COL_CATALOG else None
    title = values[COL_TITLE - 1] if len(values) >= COL_TITLE else None
    rel = values[COL_REL_DATE - 1] if len(values) >= COL_REL_DATE else None
    typ = values[COL_TYPE - 1] if len(values) >= COL_TYPE else None
    units = values[COL_UNITS - 1] if len(values) >= COL_UNITS else None
    ws_price = values[COL_WS - 1] if len(values) >= COL_WS else None
    base = values[COL_BASE - 1] if len(values) >= COL_BASE else None
    prorata = values[COL_PRORATA - 1] if len(values) >= COL_PRORATA else None
    ctrl = values[COL_CTRL - 1] if len(values) >= COL_CTRL else None
    rate = values[COL_RATE - 1] if len(values) >= COL_RATE else None
    gross_cell = values[COL_GROSS - 1] if len(values) >= COL_GROSS else None
    currency = values[COL_CURRENCY - 1] if len(values) >= COL_CURRENCY else None
    territory = values[COL_TERRITORY - 1] if len(values) >= COL_TERRITORY else None
    if territory in (None, ''):
        territory = territory_from_currency(currency)
    rep_year = values[COL_REP_YEAR - 1] if len(values) >= COL_REP_YEAR else None
    rep_qtr = values[COL_REP_QTR - 1] if len(values) >= COL_REP_QTR else None
    rev_year = values[COL_REV_YEAR - 1] if len(values) >= COL_REV_YEAR else None
    rev_qtr = values[COL_REV_QTR - 1] if len(values) >= COL_REV_QTR else None
    num = values[COL_NUM - 1] if len(values) >= COL_NUM else None
    den = values[COL_DEN - 1] if len(values) >= COL_DEN else None

    frac = None
    if num not in (None, '') and den not in (None, ''):
        try:
            frac = (int(float(num)), int(float(den)))
        except (TypeError, ValueError):
            frac = None
    if frac is None:
        frac = parse_fraction(prorata)
    if frac is None:
        num_n = den_n = None
        prorata_out = prorata
    else:
        num_n, den_n = frac
        prorata_out = f'{num_n}/{den_n}'

    gross = to_float(gross_cell)
    if gross is None:
        gross = compute_physical_gross(units, base, rate, num_n, den_n)

    amount = None
    try:
        if base not in (None, '') and units not in (None, ''):
            amount = float(base) * float(units)
    except (TypeError, ValueError):
        amount = None

    rev_period = None
    if rev_year not in (None, '') or rev_qtr not in (None, ''):
        rev_period = f'{cell_str(rev_year)} {cell_str(rev_qtr)}'.strip()

    mapped = {
        'Entry No.': None,
        'Payer/Licensee': payer,
        'Payee/Licensor': payee,
        'Territory': territory,
        'Report year': rep_year,
        'Report Quarter': rep_qtr,
        'Rev Year': rev_year,
        'Rev Quarter': rev_qtr,
        'USER': 'Regular Trade',
        'CATALOG NO.': catalog,
        'PayType': 'Regular Trade',
        'CATALOG TITLE': title,
        'SONG TITLE': None,
        'ARTIST': '莫文蔚',
        'REVENUE PERIOD': rev_period,
        'UNIT': units,
        'AMOUNT': amount,
        'SongProRata': prorata_out,
        'Ctrl.%': ctrl,
        'Royalty Rate%': rate,
        'ROYALTY': gross,
        'Share%': 100,
        'SHARE AMOUNT (local FX)': gross,
        'Currency': currency,
        'WS Price': ws_price,
        'Base Price': base,
        'Release Date': rel,
        'Type': typ,
    }
    return [mapped.get(h) for h in out_headers]


header('PART 2 — NewMedia + combined JN file')
path_physical = physical_output_path
print(f'  Physical file (output folder only): {path_physical}')
if not os.path.isfile(path_physical):
    close_log()
    raise FileNotFoundError(
        'Physical output not found in the output folder. '
        f'Part 1 should have written: {path_physical}'
    )

header('6. Read NewMedia workbooks')
fx_store = OrderedDict()
sheet_results = []

for path, fname in [(path_newmedia, file_newmedia)]:
    subheader(fname)
    wb = load_workbook(path, read_only=True, data_only=True)
    print(f'  Sheets                   : {wb.sheetnames}')
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        result = process_sheet(ws, sheet_name, fname, fx_store)
        sheet_results.append(result)
        print(
            f'  {sheet_name:<16} territory={result["territory"]:<10} '
            f'USER row={result["header_row_idx"]}  '
            f'dropped above USER={fmt_int(result["n_above_user"])}  '
            f'dropped empty A={fmt_int(result["n_empty_a"])}  '
            f'data rows={fmt_int(len(result["data_rows"]))}'
        )
        print(f'    Payer/Licensee         : {result["payer"]}')
        print(f'    Payee/Licensor         : {result["payee"]}')
        if result['header_row_idx'] is None:
            wb.close()
            close_log()
            raise ValueError(f'No USER header in column A of {fname} / {sheet_name}')
        if not result['payer'] or not result['payee']:
            wb.close()
            close_log()
            raise ValueError(f'Missing Payer/Payee in {fname} / {sheet_name}')
    wb.close()

needed = {p.split(':')[1].split('/')[0] for p in CURRENCY_PAIRS}
missing_fx = sorted(needed - set(fx_store))
if missing_fx:
    print(f'\n  FX not found in the Excel files: {missing_fx}')
    print('  Searching PDFs in the input folder for the missing rates...')
    scan_pdfs_for_fx(quarterly_dir, fx_store)
    missing_fx = sorted(needed - set(fx_store))

header('7. Update Rock_lookup_fx.csv')
print('  Rates found:')
for code in FX_WRITE_ORDER:
    rec = fx_store.get(code)
    if rec is None:
        print(f'    {code:<4}  MISSING')
        continue
    print(
        f'    {code:<4}  {rec["pair"]:<18}  rate={format_rate(rec["rate"]):<14}  '
        f'from {rec["source"]}'
    )

if missing_fx:
    close_log()
    raise ValueError(
        'Could not find FX rates for: ' + ', '.join(missing_fx)
        + '. Expected a Currency:XXX/HKD cell with a ratio in the next cell.'
    )

with open(path_fx, 'r', encoding='utf-8-sig', newline='') as f:
    raw_lines = f.read().splitlines()
if not raw_lines:
    close_log()
    raise ValueError(f'Empty FX lookup: {path_fx}')

header_line = raw_lines[0]
kept = []
removed = 0
for line in raw_lines[1:]:
    if not line.strip():
        continue
    parts = next(csv.reader([line]))
    if (
        len(parts) >= 3
        and str(parts[1]).strip() == str(REPORT_YEAR)
        and str(parts[2]).strip() == REPORT_QUARTER
    ):
        removed += 1
        continue
    kept.append(line)

new_fx_lines = []
for code in FX_WRITE_ORDER:
    rec = fx_store[code]
    key = f'{code}{REPORT_YEAR}{REPORT_QUARTER}'
    row = [''] * 13
    row[0] = code
    row[1] = str(REPORT_YEAR)
    row[2] = REPORT_QUARTER
    row[3] = format_rate(rec['rate'])
    row[4] = key
    new_fx_lines.append(','.join(row))

out_text = '\r\n'.join([header_line] + kept + new_fx_lines) + '\r\n'
with open(path_fx, 'w', encoding='utf-8-sig', newline='') as f:
    f.write(out_text)

print(f'  Removed existing {REPORT_YEAR} {REPORT_QUARTER} rows : {fmt_int(removed)}')
print(f'  New FX rows written              : {fmt_int(len(new_fx_lines))}')
print(f'  Lookup saved                     : {path_fx}')
for line in new_fx_lines:
    print(f'    {line}')

header('8. Combine cleaned sheets')
base_headers = list(SOURCE_COLUMNS)
for result in sheet_results:
    print(f'  {result["sheet_name"]:<16} source columns: {result["source_headers"]}')

out_headers = NEW_HEADERS + base_headers + EXTRA_HEADERS
write_headers = out_headers + ['Royalty (HKD)']
print(f'  Output columns           : {fmt_int(len(write_headers))}')
print(f'  {write_headers}')
print('  Royalty (HKD)            : SHARE AMOUNT (local FX) / FX rate  (local per HKD, same as Rock matching)')

FX_CODE_ALIASES = {'MAL': 'MYR'}
fx_missing_counts = {}


def fx_code_for_currency(currency):
    code = cell_str(currency).upper()
    return FX_CODE_ALIASES.get(code, code)


def royalty_hkd_from_row(row):
    try:
        share_i = out_headers.index('SHARE AMOUNT (local FX)')
        ccy_i = out_headers.index('Currency')
    except ValueError:
        return None
    share = row[share_i] if share_i < len(row) else None
    currency = row[ccy_i] if ccy_i < len(row) else None
    try:
        share_n = float(share)
    except (TypeError, ValueError):
        return None
    code = fx_code_for_currency(currency)
    rec = fx_store.get(code)
    if rec is None or not rec.get('rate'):
        key = code or '(blank)'
        fx_missing_counts[key] = fx_missing_counts.get(key, 0) + 1
        return None
    return share_n / rec['rate']


def append_output_row(row):
    royalty = royalty_hkd_from_row(row)
    ws_out.append(list(row) + [royalty])
    return royalty


wb_out = Workbook(write_only=True)
ws_out = wb_out.create_sheet('data')
ws_out.append(write_headers)

total_rows = 0
total_royalty_hkd = 0.0
unit_i = out_headers.index('UNIT')
amount_i = out_headers.index('AMOUNT')
share_i = out_headers.index('SHARE AMOUNT (local FX)')
for result in sheet_results:
    n = len(result['data_rows'])
    units = 0.0
    amount = 0.0
    share = 0.0
    royalty = 0.0
    for row in result['data_rows']:
        r_hkd = append_output_row(row)
        try:
            units += float(row[unit_i] or 0)
        except (TypeError, ValueError):
            pass
        try:
            amount += float(row[amount_i] or 0)
        except (TypeError, ValueError):
            pass
        try:
            share += float(row[share_i] or 0)
        except (TypeError, ValueError):
            pass
        if r_hkd is not None:
            royalty += r_hkd
    total_rows += n
    total_royalty_hkd += royalty
    print(
        f'  {result["sheet_name"]:<16} {result["territory"]:<10}  '
        f'rows {fmt_int(n):>10}    UNIT {fmt_int(units):>14}    '
        f'AMOUNT {fmt_money(amount):>14}    SHARE {fmt_money(share):>14}    '
        f'Royalty HKD {fmt_money(royalty):>14}'
    )

header('9. Append physical rows from output-folder data tab')
print(f'  File                     : {path_physical}')
print(f'  Sheet                    : {DATA_SHEET}')
print(f'  Filter                   : Report year={REPORT_YEAR}, Report Quarter={REPORT_QUARTER}')
print('  Source                   : physical `data` tab (computed; maforma formulas have no Excel cache yet)')

wb_phys = load_workbook(path_physical, read_only=True, data_only=False)
if DATA_SHEET not in wb_phys.sheetnames:
    wb_phys.close()
    close_log()
    raise KeyError(f'Sheet "{DATA_SHEET}" not found. Sheets: {wb_phys.sheetnames}')
ws_phys = wb_phys[DATA_SHEET]


def same_period(year, quarter):
    return str(year).strip() == str(REPORT_YEAR) and str(quarter).strip() == str(REPORT_QUARTER)


physical_rows = []
n_phys_read = 0
phys_payer = sheet_results[0]['payer']
phys_payee = sheet_results[0]['payee']
print(f'  Payer/Licensee           : {phys_payer}')
print(f'  Payee/Licensor           : {phys_payee}')
for row in ws_phys.iter_rows(min_row=2, max_col=23, values_only=True):
    n_phys_read += 1
    year = row[COL_REP_YEAR - 1] if len(row) >= COL_REP_YEAR else None
    quarter = row[COL_REP_QTR - 1] if len(row) >= COL_REP_QTR else None
    if same_period(year, quarter):
        physical_rows.append(physical_data_row_to_output(row, out_headers, phys_payer, phys_payee))
wb_phys.close()

phys_units = 0.0
phys_share = 0.0
phys_royalty = 0.0
unit_out_idx = out_headers.index('UNIT') if 'UNIT' in out_headers else None
share_out_idx = out_headers.index('SHARE AMOUNT (local FX)') if 'SHARE AMOUNT (local FX)' in out_headers else None
for row in physical_rows:
    r_hkd = append_output_row(row)
    if unit_out_idx is not None:
        try:
            phys_units += float(row[unit_out_idx] or 0)
        except (TypeError, ValueError):
            pass
    if share_out_idx is not None:
        try:
            phys_share += float(row[share_out_idx] or 0)
        except (TypeError, ValueError):
            pass
    if r_hkd is not None:
        phys_royalty += r_hkd

print(f'  data rows scanned        : {fmt_int(n_phys_read)}')
print(f'  Matching period rows     : {fmt_int(len(physical_rows))}')
print(f'  UNIT                     : {fmt_int(phys_units)}')
print(f'  SHARE AMOUNT             : {fmt_money(phys_share)}')
print(f'  Royalty (HKD)            : {fmt_money(phys_royalty)}')
total_rows += len(physical_rows)
total_royalty_hkd += phys_royalty
if fx_missing_counts:
    print('  WARNING: no FX rate for:')
    for code, n in sorted(fx_missing_counts.items()):
        print(f'    {code:<8} {fmt_int(n)} row(s)')

wb_out.save(newmedia_output_path)
wb_out.close()

with zipfile.ZipFile(newmedia_output_path, 'r') as zf:
    names = zf.namelist()
required_zip = ['[Content_Types].xml', 'xl/workbook.xml', 'xl/worksheets/sheet1.xml']
missing_zip = [n for n in required_zip if n not in names]
if missing_zip:
    close_log()
    raise RuntimeError(f'Output xlsx is missing {missing_zip}')
print(f'  xlsx package check       : OK ({fmt_int(len(names))} zip entries)')

header('10. Compare Royalty (HKD) to 總表 PDF')
print(f'  PDF                      : {path_summary_pdf}')
SUMMARY_DETAIL_RE = re.compile(
    r'(?P<name>.+?)\s+'
    r'(?P<local_open>\()?'
    r'(?P<local_ccy>HKD|NTD|USD|RMB|SGD|YEN|MYR|CNY)'
    r'(?P<local>[\d,]+\.\d{2})'
    r'(?P<local_close>\))?'
    r'\s+'
    r'(?P<gross_open>\()?'
    r'HKD(?P<gross>[\d,]+\.\d{2})'
    r'(?P<gross_close>\))?'
    r'\s*'
    r'(?P<tax>[\d.]+)\s*%\s+'
    r'(?P<net_open>\()?'
    r'HKD(?P<net>[\d,]+\.\d{2})'
    r'(?P<net_close>\))?',
    re.I,
)


def signed_amount(match, amount_group, open_group, close_group):
    amount = float(match.group(amount_group).replace(',', ''))
    if match.group(open_group) or match.group(close_group):
        amount = -amount
    return amount


reader = PdfReader(path_summary_pdf)
pdf_text = '\n'.join((page.extract_text() or '') for page in reader.pages)
pdf_rows = []
for m in SUMMARY_DETAIL_RE.finditer(pdf_text):
    pdf_rows.append({
        'name': cell_str(m.group('name')),
        'local_ccy': m.group('local_ccy').upper(),
        'local': signed_amount(m, 'local', 'local_open', 'local_close'),
        'gross_hkd': signed_amount(m, 'gross', 'gross_open', 'gross_close'),
        'tax_pct': m.group('tax'),
        'net_hkd': signed_amount(m, 'net', 'net_open', 'net_close'),
    })

if not pdf_rows:
    close_log()
    raise ValueError(
        f'No G.ROYALTY(HKD$) lines found in {summary_pdf}. '
        'Check that the PDF still uses columns G.ROYALTY / G.ROYALTY(HKD$).'
    )

pdf_royalty_hkd = sum(r['gross_hkd'] for r in pdf_rows)
print(f'  G.ROYALTY(HKD$) lines    : {fmt_int(len(pdf_rows))}')
for r in pdf_rows:
    print(
        f'    {r["name"]:<28} {r["local_ccy"]:<4} {fmt_money(r["local"]):>14}    '
        f'G.ROYALTY(HKD$) {fmt_money(r["gross_hkd"]):>14}'
    )

diff = total_royalty_hkd - pdf_royalty_hkd
print()
print(f'  Sum G.ROYALTY(HKD$) PDF  : {fmt_money(pdf_royalty_hkd)}')
print(f'  Sum Royalty (HKD) xls    : {fmt_money(total_royalty_hkd)}')
print(f'  Difference (xls - PDF)   : {fmt_money(diff)}')
if abs(diff) > 0.01:
    print('  ALERT: difference is larger than HKD 0.01')
else:
    print('  Check                      : OK (difference <= HKD 0.01)')

header('11. Output summary')
print(f'  Physical workbook        : {physical_output_path}')
print(f'  Combined workbook        : {newmedia_output_path}')
print(f'  NewMedia data rows       : {fmt_int(total_rows - len(physical_rows))}')
print(f'  Physical data rows       : {fmt_int(len(physical_rows))}')
print(f'  Total data rows          : {fmt_int(total_rows)}')
print(f'  Royalty (HKD) total      : {fmt_money(total_royalty_hkd)}')
print(f'  FX lookup                : {path_fx}  (+{fmt_int(len(new_fx_lines))} rows for {REPORT_YEAR} {REPORT_QUARTER})')
print(f'  Run finished             : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
close_log()



  PART 2 — NewMedia + combined JN file
  Physical file (output folder only): /Users/johannesnatterer/Developer/_output/ZZ_Rock_royalties_physical_2024Q1_2026Q2.xlsx

  6. Read NewMedia workbooks

--- MOK 2026Q2_NewMeidia.xlsx ---
  Sheets                   : ['2026Q2_TWN', '2026Q2_HK']
  2026Q2_TWN       territory=Taiwan     USER row=5  dropped above USER=4  dropped empty A=29  data rows=195,639
    Payer/Licensee         : ROCK(TWN)滾石台灣
    Payee/Licensor         : MOK-A-BYE BABY MUSIC LIMITED(莫文蔚)
  2026Q2_HK        territory=Hong Kong  USER row=5  dropped above USER=4  dropped empty A=5  data rows=200
    Payer/Licensee         : ROCK(TWN)滾石台灣
    Payee/Licensor         : MOK-A-BYE BABY MUSIC LIMITED(莫文蔚)

  FX not found in the Excel files: ['MYR']
  Searching PDFs in the input folder for the missing rates...
  [SKIP]     2026Q2 MOK-TWN_ NewMedia.pdf  (22.8 MB > 1 MB)

  7. Update Rock_lookup_fx.csv
  Rates found:
    NTD   31.85:7.843         rate=4.060946067     from MOK 2026Q2_N